- examples
    - https://blog.langchain.com/enhancing-rag-based-applications-accuracy-by-constructing-and-leveraging-knowledge-graphs/

### raw documents

In [1]:
from langchain.document_loaders import WikipediaLoader

In [2]:
raw_documents = WikipediaLoader(query="Elizabeth I").load()

/home/zhangchunhui/miniconda3/envs/verl/lib/python3.10/site-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /home/zhangchunhui/miniconda3/envs/verl/lib/python3.10/site-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


In [6]:
from langchain.text_splitter import TokenTextSplitter
# Define chunking strategy
text_splitter = TokenTextSplitter(chunk_size=512, chunk_overlap=24)
documents = text_splitter.split_documents(raw_documents[:3])

### neo4j

In [1]:
# from langchain_community.graphs import Neo4jGraph
from langchain_neo4j import Neo4jGraph

In [5]:
graph = Neo4jGraph(
    url="neo4j://192.168.194.175:7687",
    username="neo4j",
    password="Apple1105,",
    refresh_schema=False
)

/tmp/ipykernel_823516/3198295067.py:1: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-neo4j package and should be used instead. To use it run `pip install -U :class:`~langchain-neo4j` and import as `from :class:`~langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(


### graph transformer 

In [7]:
from dotenv import load_dotenv, find_dotenv
assert load_dotenv(find_dotenv())

In [8]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(model='gpt-4.1-mini', model_provider='openai')

In [9]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
llm_transformer = LLMGraphTransformer(llm=llm)

In [11]:
graph_documents = llm_transformer.convert_to_graph_documents(documents)

In [12]:
# Store to neo4j
graph.add_graph_documents(
  graph_documents, 
  baseEntityLabel=True, 
  include_source=True
)